# B04 — Advanced ETL Patterns and Orchestration

**Time: about 75 minutes.**
Covers: Advanced ETL Patterns · Orchestrating Batch Pipelines

### What you will be able to do afterwards

- Build a Type 2 slowly changing dimension and defend its invariants.
- Write a backfill that is safe to run twice.
- Keep a run log so a pipeline knows what it has already done.
- Wire notebooks into a multi-task Job with a Databricks Asset Bundle.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils

cfg = utils.get_configs()
catalog, gold_schema = cfg["catalog"], cfg["schema_gold"]

products_raw = f"{cfg['catalog']}.{cfg['schema_bronze']}.products"
dim_product = utils.get_configs("dim_product")["table_gold"]
fact_sales = utils.get_configs("fact_sales_daily")["table_gold"]
run_log = utils.get_configs("pipeline_runs")["table_gold"]
sales_silver = utils.get_configs("sales")["table_silver"]

spark = utils.spark
print(dim_product, fact_sales, run_log, sep="\n")

## Step 0 — Load the product snapshots

You need the daily product files as history, not just the latest state.

**TO DO**

Load every `products_*.csv` into a bronze table with a `snapshot_date` column derived from
the file name. One row per product per day.

In [ ]:
# TO DO: load the product snapshots with a snapshot_date column

## Step 1 — A Type 2 dimension

A Type 1 dimension overwrites. A Type 2 keeps every version with a validity interval, so a
sale from March joins to the price that was live in March.

**TO DO**

Build `dim_product` in gold:

| Column | Type | Notes |
| :-- | :-- | :-- |
| `surrogate_key` | `STRING` | unique per version, not per product |
| `product_id` | `INT` | the business key |
| `name`, `category`, `price` | | the tracked attributes |
| `valid_from` | `DATE` | first snapshot where this version appeared |
| `valid_to` | `DATE` | day before the next version; `NULL` while current |
| `is_current` | `BOOLEAN` | |

A new version is opened whenever any tracked attribute changes between consecutive
snapshots for a product.

**Tips**

- `LAG(...) OVER (PARTITION BY product_id ORDER BY snapshot_date)` finds the changes.
- Once you have the change points, `LEAD` on `valid_from` gives you `valid_to`.
- `surrogate_key` can be a hash of `product_id` and `valid_from`.

**Three invariants your result must satisfy** — the checks test all three:

1. Exactly one row per `product_id` with `is_current = true`.
2. At least one product has more than one row.
3. For a given product, no two validity intervals overlap.

> **Question:** invariant 3 is the one people get wrong. What does an overlap do to a
> revenue report that joins sales to the dimension on date?

In [ ]:
def build_dim_product(source: str, target: str) -> None:
    """
    Build a Type 2 dimension from daily product snapshots.

    Args:
        source: bronze table with one row per product per snapshot_date.
        target: gold dimension table to create or replace.
    """
    # TO DO
    pass


# TO DO: build it, then check all three invariants yourself before running the checks

## Step 2 — A backfill you can run twice

**TO DO**

Build `fact_sales_daily` in gold: one row per `sale_date` and `product_id`, with
`total_quantity`, `total_amount` and `num_sales`, from the active rows of silver sales.

Then write `load_day(date)` that loads **one day** and is safe to re-run:

```python
(df.write
   .format("delta")
   .mode("overwrite")
   .option("replaceWhere", f"sale_date = '{date}'")
   .saveAsTable(fact_sales))
```

`replaceWhere` replaces only the rows matching the predicate. Re-running the same day
overwrites its own output instead of appending a second copy.

**TO DO**

1. Load every day.
2. Record the row count.
3. Re-run **one** day.
4. Confirm the row count is unchanged and no `(sale_date, product_id)` pair is duplicated.

> **Questions:**
> - What happens if the predicate in `replaceWhere` doesn't match the data being written?
> - `replaceWhere` is one way to make a load idempotent. `MERGE` is another. When would you
>   pick each?

In [ ]:
def load_day(target: str, source: str, sale_date: str) -> int:
    """
    Load exactly one day into the fact table, replacing anything already there for that day.

    Args:
        target: gold fact table.
        source: silver sales table.
        sale_date: 'YYYY-MM-DD'.
    Returns:
        Rows written.
    """
    # TO DO
    pass


# TO DO: load all days, count, re-run one day, count again

## Step 3 — Let the pipeline remember

**TO DO**

Create `pipeline_runs` in gold with `run_id`, `run_timestamp`, `processed_date`,
`rows_written`, `status`, `error_message`.

Then:

1. Wrap `load_day` so every call appends a row to the log, including failures.
2. Write `days_to_process()` that returns the dates present in the source but with no
   successful run in the log.
3. Re-run your loader. It should now process nothing.
4. Force a re-run of one day anyway, and confirm the log records it twice while the fact
   table stays the same size.

> **Question:** the log is in the same catalog as the data it describes. If the whole
> catalog is unavailable, the pipeline cannot find out what it has done. Is that acceptable?
> What would you do differently for a pipeline that matters?

In [ ]:
def load_day_logged(target: str, source: str, sale_date: str, log_table: str) -> None:
    """Run load_day and record the outcome, success or failure."""
    # TO DO
    pass


def days_to_process(source: str, log_table: str) -> list:
    """Dates in source with no successful run in the log."""
    # TO DO
    pass


# TO DO: create the log, run, re-run, force one day, inspect

## Step 4 — Orchestrate it

Everything so far ran interactively. Now schedule it.

The `project_bundle/` folder in this repo holds a Databricks Asset Bundle. `databricks.yml`
and the cluster resource are complete. The job definition is partly complete: the first
task is filled in as a worked example, the second and third are yours.

**TO DO**

1. Open `project_bundle/resources/batch_pipeline.job.yml`.
2. Complete the two remaining tasks so the job runs B01 → B04 in order, with the correct
   `depends_on` edges.
3. Pass `catalog` as a job parameter and read it in the notebooks with
   `utils.get_param("catalog", ...)` — it is already wired up.
4. Deploy and run:

   ```bash
   databricks bundle validate
   databricks bundle deploy --target dev
   databricks bundle run batch_pipeline_job
   ```

   If you have no CLI access, build the same job in the Jobs UI instead and paste a
   screenshot into your submission.

> **Questions:**
> - The bundle names the job `${var.env}_batch_pipeline_${workspace.current_user.short_name}`.
>   Why include the username, given every student deploys to the same workspace?
> - Your job has three tasks in a chain. Task 2 fails. What is the retry story — do you
>   re-run the whole job or just task 2, and what does your answer require of task 2?
> - `mode: development` pauses schedules and prefixes resource names. What breaks if you
>   deploy with `mode: production` to a shared workspace by accident?

In [ ]:
# TO DO: nothing to run here — edit the YAML, deploy, and record the run URL below.
#
# Job run URL:

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("B04-advanced-etl-and-orchestration")

## Recap

- A Type 2 dimension is defined by its invariants, not by its columns. Test the invariants.
- `replaceWhere` makes a per-partition load idempotent without a merge.
- A run log turns "did that day load?" from an investigation into a query.
- A bundle puts the orchestration in the repo, reviewed like everything else, instead of in
  somebody's browser tab.